# 8 - Text Splitting no LangChain

Text splitting é o processo de dividir textos longos em pedaços menores (chunks) para processamento eficiente por LLMs.

Os principais splitters disponíveis:
- `CharacterTextSplitter` — divide por um separador fixo
- `RecursiveCharacterTextSplitter` — tenta separadores em cascata, respeitando estrutura do texto
- `TokenTextSplitter` — divide com base em tokens reais (usando tiktoken)

> **Padrão moderno**: todos os splitters estão em `langchain_text_splitters` (pacote separado `langchain-text-splitters`).

## Texto de exemplo

In [6]:
texto_completo = """
Python é uma linguagem de programação de alto nível conhecida por sua simplicidade,
legibilidade e versatilidade. Seu design foi criado com o objetivo de ser fácil de
aprender e usar, permitindo que programadores escrevam código de maneira mais
intuitiva e eficiente. Ao contrário de outras linguagens, Python prioriza a
legibilidade do código, o que facilita a compreensão e manutenção do software,
mesmo por programadores que não são os autores do código original.
Sua sintaxe clara, por exemplo, elimina a necessidade de muitos símbolos ou
palavras-chave complicadas, tornando o código mais próximo da linguagem humana.
"""

In [2]:
from langchain_text_splitters import CharacterTextSplitter

import string
texto = "".join(f"{string.ascii_lowercase}" for _ in range(5))

print(f"Tamanho do texto: {len(texto)} caracteres")
print(texto)

Tamanho do texto: 130 caracteres
abcdefghijklmnopqrstuvwxyzabcdefghijklmnopqrstuvwxyzabcdefghijklmnopqrstuvwxyzabcdefghijklmnopqrstuvwxyzabcdefghijklmnopqrstuvwxyz


In [3]:
# Sem separator explícito: usa '\n\n' por padrão → texto não é dividido
chunk_size = 50
chunk_overlap = 0

char_split = CharacterTextSplitter(
    chunk_size=chunk_size,
    chunk_overlap=chunk_overlap,
)

split = char_split.split_text(texto)
print(f"Número de chunks: {len(split)}")
split

Número de chunks: 1


['abcdefghijklmnopqrstuvwxyzabcdefghijklmnopqrstuvwxyzabcdefghijklmnopqrstuvwxyzabcdefghijklmnopqrstuvwxyzabcdefghijklmnopqrstuvwxyz']

In [4]:
# Com separator="": divide caractere a caractere → respeita chunk_size
char_split = CharacterTextSplitter(
    chunk_size=chunk_size,
    chunk_overlap=chunk_overlap,
    separator="",
)

split = char_split.split_text(texto)
print(f"Número de chunks: {len(split)}")
split

Número de chunks: 3


['abcdefghijklmnopqrstuvwxyzabcdefghijklmnopqrstuvwx',
 'yzabcdefghijklmnopqrstuvwxyzabcdefghijklmnopqrstuv',
 'wxyzabcdefghijklmnopqrstuvwxyz']

In [7]:
# Aplicando no texto_completo
char_split = CharacterTextSplitter(
    chunk_size=50,
    chunk_overlap=0,
    separator="",
)

split = char_split.split_text(texto_completo)
print(f"Número de chunks: {len(split)}")
split

Número de chunks: 13


['Python é uma linguagem de programação de alto nív',
 'el conhecida por sua simplicidade,\nlegibilidade e',
 'versatilidade. Seu design foi criado com o objetiv',
 'o de ser fácil de\naprender e usar, permitindo que',
 'programadores escrevam código de maneira mais\nintu',
 'itiva e eficiente. Ao contrário de outras linguage',
 'ns, Python prioriza a\nlegibilidade do código, o qu',
 'e facilita a compreensão e manutenção do software,',
 'mesmo por programadores que não são os autores do',
 'código original.\nSua sintaxe clara, por exemplo,',
 'elimina a necessidade de muitos símbolos ou\npalavr',
 'as-chave complicadas, tornando o código mais próxi',
 'mo da linguagem humana.']

## 2. RecursiveCharacterTextSplitter

Mais sofisticado: tenta uma **lista de separadores em ordem**, do mais amplo ao mais estreito.

Separadores padrão: `["\n\n", "\n", " ", ""]`

Isso preserva melhor a estrutura semântica do texto — parágrafos antes de frases, frases antes de palavras.

In [8]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

# Texto simples (sem separadores naturais)
chunk_size = 50
chunk_overlap = 10

recursive_split = RecursiveCharacterTextSplitter(
    chunk_size=chunk_size,
    chunk_overlap=chunk_overlap,
    separators=[".", " ", ""],
)

split = recursive_split.split_text(texto)
print(f"Número de chunks: {len(split)}")
split

Número de chunks: 3


['abcdefghijklmnopqrstuvwxyzabcdefghijklmnopqrstuvwx',
 'opqrstuvwxyzabcdefghijklmnopqrstuvwxyzabcdefghijkl',
 'cdefghijklmnopqrstuvwxyzabcdefghijklmnopqrstuvwxyz']

In [9]:
# Texto completo: separadores em cascata respeitam parágrafos e frases
chunk_size = 250
chunk_overlap = 10

recursive_split = RecursiveCharacterTextSplitter(
    chunk_size=chunk_size,
    chunk_overlap=chunk_overlap,
    separators=["\n\n", "\n", " ", ""],
)

split = recursive_split.split_text(texto_completo)
print(f"Número de chunks: {len(split)}")
for i, chunk in enumerate(split):
    print(f"\n--- Chunk {i+1} ({len(chunk)} chars) ---")
    print(chunk)

Número de chunks: 3

--- Chunk 1 (244 chars) ---
Python é uma linguagem de programação de alto nível conhecida por sua simplicidade,
legibilidade e versatilidade. Seu design foi criado com o objetivo de ser fácil de
aprender e usar, permitindo que programadores escrevam código de maneira mais

--- Chunk 2 (221 chars) ---
intuitiva e eficiente. Ao contrário de outras linguagens, Python prioriza a
legibilidade do código, o que facilita a compreensão e manutenção do software,
mesmo por programadores que não são os autores do código original.

--- Chunk 3 (155 chars) ---
Sua sintaxe clara, por exemplo, elimina a necessidade de muitos símbolos ou
palavras-chave complicadas, tornando o código mais próximo da linguagem humana.


## 3. TokenTextSplitter

Divide com base em **tokens** reais (usando [tiktoken](https://github.com/openai/tiktoken)) em vez de caracteres.

Por que isso importa? LLMs têm limite de contexto em **tokens**, não em caracteres. Um token equivale aproximadamente a 4 caracteres em inglês.

> **Dependência**: requer o pacote `tiktoken` instalado (`pip install tiktoken`).

In [10]:
from langchain_text_splitters import TokenTextSplitter

chunk_size = 50
chunk_overlap = 5

token_split = TokenTextSplitter(
    chunk_size=chunk_size,
    chunk_overlap=chunk_overlap,
    # encoding_name="cl100k_base"  # padrão GPT-4 / GPT-3.5 / text-embedding-ada-002
)

split = token_split.split_text(texto)
print(f"Número de chunks: {len(split)}")
split

Número de chunks: 2


['abcdefghijklmnopqrstuvwxyzabcdefghijklmnopqrstuvwxyzabcdefghijklmnopqrstuvwxyzabcdefghijklmnopq',
 'ijklmnopqrstuvwxyzabcdefghijklmnopqrstuvwxyz']

## 4. Splitting de documentos PDF

Na prática, você carrega documentos reais e os divide para indexação em vetores.

O fluxo típico é:
1. **Carregar** com um `DocumentLoader` (ex: `PyPDFLoader`)
2. **Dividir** com um `TextSplitter`
3. **Indexar** em um vector store

> **Import moderno**: `PyPDFLoader` está em `langchain_community.document_loaders`.
> Requer: `pip install langchain-community pypdf`

In [11]:
from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter

chunk_size = 50
chunk_overlap = 10

recursive_split = RecursiveCharacterTextSplitter(
    chunk_size=chunk_size,
    chunk_overlap=chunk_overlap,
)

In [12]:
arquivo = "../files/apostila.pdf"
loader = PyPDFLoader(arquivo)
docs = loader.load()

print(f"Total de páginas: {len(docs)}")
print("\nPrimeiro documento:")
docs[0]

Total de páginas: 28

Primeiro documento:


Document(metadata={'producer': 'Microsoft® Word 2013', 'creator': 'Microsoft® Word 2013', 'creationdate': '2016-05-04T10:06:39-03:00', 'author': 'lucas', 'moddate': '2016-05-04T10:06:39-03:00', 'source': '../files/apostila.pdf', 'total_pages': 28, 'page': 0, 'page_label': '1'}, page_content='INTRODUÇÃO À PROGRAMAÇÃO \nCOM PYTHON \n \n \n \n \n \n \n \nPrograma de Educação Tutorial \nGrupo PET - ADS   \nIFSP -  Câmpus São Carlos')

In [13]:
# split_documents preserva os metadados (source, page) de cada Document
split = recursive_split.split_documents(docs)
print(f"Total de chunks: {len(split)}")
print("\nExemplo do primeiro chunk:")
print(split[0])

Total de chunks: 1101

Exemplo do primeiro chunk:
page_content='INTRODUÇÃO À PROGRAMAÇÃO 
COM PYTHON' metadata={'producer': 'Microsoft® Word 2013', 'creator': 'Microsoft® Word 2013', 'creationdate': '2016-05-04T10:06:39-03:00', 'author': 'lucas', 'moddate': '2016-05-04T10:06:39-03:00', 'source': '../files/apostila.pdf', 'total_pages': 28, 'page': 0, 'page_label': '1'}


## Resumo

| Splitter | Quando usar |
|---|---|
| `CharacterTextSplitter` | Textos simples com separador conhecido |
| `RecursiveCharacterTextSplitter` | **Recomendado para a maioria dos casos** — respeita estrutura do texto |
| `TokenTextSplitter` | Quando o limite de contexto do modelo é crítico |

**Dica**: prefira sempre o `RecursiveCharacterTextSplitter` como ponto de partida, ajustando `chunk_size` e `chunk_overlap` conforme o modelo e o caso de uso.